# MAE: Masked Autoencoders Are Scalable Vision Learners

**Paper**: He et al., 2021 (Facebook AI)

MAE is a **reconstruction-based** self-supervised method inspired by BERT for NLP. The key idea is simple: mask a large fraction (75%) of image patches, run a **ViT encoder only on the visible patches**, then use a **lightweight decoder** to reconstruct the original pixel values of masked patches.

Why does this work?
- Images have far more spatial redundancy than text — so masking must be aggressive (75%) to make the task non-trivial
- Running the encoder **only on visible patches** (25%) makes training efficient: full ViT on 196 patches → encoder on 49 patches
- The decoder is intentionally shallow and narrow — forcing all semantic information into the encoder
- No augmentation tricks, contrastive pairs, or momentum encoders needed

```
Image (196 patches)
  ↓ random mask 75%
Visible patches (49) → Encoder (deep ViT) → latent tokens
  + mask tokens (147)  → Decoder (shallow) → pixel reconstruction
                                               ↑ MSE loss on masked patches only
```

<img src="../figures/mae_arch.png" width="800"/>

*MAE architecture: encoder processes only visible patches (25%), decoder reconstructs masked patches (75%).*

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
import numpy as np
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## Patch Embedding

Following ViT, we split the image into non-overlapping patches of size `patch_size × patch_size` and linearly project each flattened patch to an embedding dimension `d_model`. For CIFAR-10 (32×32) with `patch_size=4`, we get `(32/4)² = 64` patches.

In [ ]:
class PatchEmbed(nn.Module):
    """Split image into patches and embed them."""

    def __init__(self, img_size=32, patch_size=4, in_ch=3, embed_dim=192):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2
        self.patch_size = patch_size
        # Convolution with kernel=stride=patch_size implements patch extraction + projection
        self.proj = nn.Conv2d(in_ch, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # x: (N, C, H, W) → (N, n_patches, embed_dim)
        x = self.proj(x)           # (N, embed_dim, H//p, W//p)
        x = x.flatten(2)           # (N, embed_dim, n_patches)
        x = x.transpose(1, 2)      # (N, n_patches, embed_dim)
        return x

## Sinusoidal 2D Positional Encoding

We use fixed sinusoidal positional encodings (not learned). The 2D position is encoded by concatenating row and column sinusoidal encodings. This is important because:
1. After masking, patch positions are shuffled — the encoder must know WHERE each visible patch came from
2. The decoder needs full positional info for all 196 positions to reconstruct the right pixels

In [ ]:
def get_2d_sincos_pos_embed(embed_dim, grid_size):
    """
    Create 2D sinusoidal positional embedding.
    Returns: (grid_size**2, embed_dim)
    """
    grid_h = np.arange(grid_size, dtype=np.float32)
    grid_w = np.arange(grid_size, dtype=np.float32)
    grid_w, grid_h = np.meshgrid(grid_w, grid_h)  # (grid_size, grid_size)

    def sincos_1d(pos, dim):
        omega = 1.0 / (10000 ** (np.arange(0, dim, 2) / dim))
        out = pos.reshape(-1, 1) * omega.reshape(1, -1)  # (N, dim//2)
        return np.concatenate([np.sin(out), np.cos(out)], axis=1)  # (N, dim)

    half = embed_dim // 2
    emb_h = sincos_1d(grid_h.flatten(), half)  # (n_patches, half)
    emb_w = sincos_1d(grid_w.flatten(), half)  # (n_patches, half)
    emb = np.concatenate([emb_h, emb_w], axis=1)  # (n_patches, embed_dim)
    return torch.tensor(emb, dtype=torch.float32)

## MAE Encoder

The encoder is a standard ViT Transformer. The key innovation: **it only processes visible (unmasked) patches**.

Steps:
1. Embed all patches
2. Add positional encodings
3. Randomly sample `mask_ratio` fraction of patches to mask
4. Pass **only the unmasked** subset through the Transformer
5. Return encoder tokens + the mask and shuffle indices (needed by decoder)

In [ ]:
class MAEEncoder(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_ch=3,
                 embed_dim=192, depth=12, num_heads=3, mlp_ratio=4.0,
                 mask_ratio=0.75):
        super().__init__()
        self.mask_ratio = mask_ratio
        self.patch_embed = PatchEmbed(img_size, patch_size, in_ch, embed_dim)
        n_patches = self.patch_embed.n_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        pos_embed = get_2d_sincos_pos_embed(embed_dim, img_size // patch_size)
        self.register_buffer('pos_embed', pos_embed.unsqueeze(0))  # (1, n_patches, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=int(embed_dim * mlp_ratio),
            dropout=0.0, activation='gelu',
            batch_first=True, norm_first=True  # Pre-LayerNorm (ViT style)
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)

        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def random_masking(self, x):
        """
        x: (N, n_patches, D)
        Returns:
          x_visible: (N, n_keep, D)
          mask: (N, n_patches) — 1=masked, 0=visible
          ids_restore: (N, n_patches) — indices to unshuffle
        """
        N, L, D = x.shape
        n_keep = int(L * (1 - self.mask_ratio))

        noise = torch.rand(N, L, device=x.device)
        ids_shuffle = noise.argsort(dim=1)            # ascending: small noise = keep
        ids_restore = ids_shuffle.argsort(dim=1)      # to undo the shuffle

        ids_keep = ids_shuffle[:, :n_keep]
        x_visible = torch.gather(x, 1, ids_keep.unsqueeze(-1).expand(-1, -1, D))

        # mask: 1 = masked, 0 = visible
        mask = torch.ones(N, L, device=x.device)
        mask[:, :n_keep] = 0
        mask = torch.gather(mask, 1, ids_restore)  # unshuffle to original order

        return x_visible, mask, ids_restore

    def forward(self, x):
        x = self.patch_embed(x)                        # (N, n_patches, D)
        x = x + self.pos_embed                         # add positional encoding
        x_vis, mask, ids_restore = self.random_masking(x)
        x_vis = self.norm(self.transformer(x_vis))     # encode visible only
        return x_vis, mask, ids_restore

## MAE Decoder

The decoder is **intentionally shallow and narrow** — e.g. 4 layers vs 12 for the encoder, 128-dim vs 192. It:
1. Projects encoder tokens to decoder dimension
2. Inserts shared **mask tokens** at masked positions (unshuffle to original order)
3. Adds full positional encodings (all patches)
4. Runs the Transformer
5. Projects to `patch_size² × 3` output (pixel values per patch)

In [ ]:
class MAEDecoder(nn.Module):
    def __init__(self, n_patches, patch_size=4, in_ch=3,
                 encoder_dim=192, decoder_dim=128,
                 depth=4, num_heads=4, mlp_ratio=4.0):
        super().__init__()
        patch_pixels = patch_size * patch_size * in_ch
        grid_size = int(math.sqrt(n_patches))

        self.embed = nn.Linear(encoder_dim, decoder_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_dim))

        pos_embed = get_2d_sincos_pos_embed(decoder_dim, grid_size)
        self.register_buffer('pos_embed', pos_embed.unsqueeze(0))  # (1, n_patches, decoder_dim)

        decoder_layer = nn.TransformerEncoderLayer(
            d_model=decoder_dim, nhead=num_heads,
            dim_feedforward=int(decoder_dim * mlp_ratio),
            dropout=0.0, activation='gelu',
            batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(decoder_layer, num_layers=depth)
        self.norm = nn.LayerNorm(decoder_dim)
        self.pred = nn.Linear(decoder_dim, patch_pixels)

        nn.init.trunc_normal_(self.mask_token, std=0.02)

    def forward(self, x_vis, ids_restore):
        """
        x_vis: (N, n_keep, encoder_dim)
        ids_restore: (N, n_patches)
        Returns: (N, n_patches, patch_pixels)
        """
        N = x_vis.size(0)
        x = self.embed(x_vis)  # (N, n_keep, decoder_dim)

        # Append mask tokens and unshuffle to original patch order
        n_masked = ids_restore.size(1) - x.size(1)
        mask_tokens = self.mask_token.expand(N, n_masked, -1)
        x_full = torch.cat([x, mask_tokens], dim=1)  # (N, n_patches, decoder_dim)

        # Unshuffle: restore original patch positions
        x_full = torch.gather(
            x_full, 1,
            ids_restore.unsqueeze(-1).expand(-1, -1, x_full.size(-1))
        )

        x_full = x_full + self.pos_embed
        x_full = self.norm(self.transformer(x_full))
        x_full = self.pred(x_full)  # (N, n_patches, patch_pixels)
        return x_full

## MAE Model + Loss

The loss is **mean squared error on masked patches only** (not the visible ones). Computing the loss on all patches would also work but the paper shows masking-only loss is better — visible patches are trivial to reconstruct.

Optionally, the paper normalizes pixel values per-patch (subtract patch mean, divide by patch std) before computing the loss — this makes patches with diverse pixel values contribute equally.

In [ ]:
class MAE(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_ch=3,
                 encoder_dim=192, encoder_depth=12, encoder_heads=3,
                 decoder_dim=128, decoder_depth=4, decoder_heads=4,
                 mask_ratio=0.75, norm_pix_loss=True):
        super().__init__()
        self.patch_size = patch_size
        self.in_ch = in_ch
        self.norm_pix_loss = norm_pix_loss

        self.encoder = MAEEncoder(
            img_size, patch_size, in_ch,
            encoder_dim, encoder_depth, encoder_heads,
            mask_ratio=mask_ratio
        )
        n_patches = self.encoder.patch_embed.n_patches

        self.decoder = MAEDecoder(
            n_patches, patch_size, in_ch,
            encoder_dim, decoder_dim,
            decoder_depth, decoder_heads
        )

    def patchify(self, imgs):
        """
        imgs: (N, 3, H, W)
        Returns: (N, n_patches, patch_size**2 * 3)
        """
        p = self.patch_size
        h = w = imgs.shape[2] // p
        x = imgs.reshape(imgs.shape[0], self.in_ch, h, p, w, p)
        x = x.permute(0, 2, 4, 3, 5, 1)  # (N, h, w, p, p, C)
        x = x.reshape(imgs.shape[0], h * w, p * p * self.in_ch)
        return x

    def forward(self, imgs):
        x_vis, mask, ids_restore = self.encoder(imgs)
        pred = self.decoder(x_vis, ids_restore)  # (N, n_patches, patch_pixels)

        # Reconstruction target: patchified pixels
        target = self.patchify(imgs)  # (N, n_patches, patch_pixels)

        if self.norm_pix_loss:
            mean = target.mean(dim=-1, keepdim=True)
            var  = target.var(dim=-1, keepdim=True)
            target = (target - mean) / (var + 1e-6).sqrt()

        loss = (pred - target) ** 2          # (N, n_patches, patch_pixels)
        loss = loss.mean(dim=-1)             # (N, n_patches)
        loss = (loss * mask).sum() / mask.sum()  # mean over masked patches only
        return loss, pred, mask


model = MAE(
    img_size=32, patch_size=4, in_ch=3,
    encoder_dim=192, encoder_depth=6, encoder_heads=3,
    decoder_dim=128, decoder_depth=4, decoder_heads=4,
    mask_ratio=0.75, norm_pix_loss=True,
).to(device)

enc_params = sum(p.numel() for p in model.encoder.parameters())
dec_params = sum(p.numel() for p in model.decoder.parameters())
print(f'Encoder params: {enc_params:,}')
print(f'Decoder params: {dec_params:,}  ({100*dec_params/enc_params:.1f}% of encoder)')

## Training

We train with **AdamW** and a cosine LR schedule. Key hyperparameters:
- `mask_ratio = 0.75` — 75% of patches are masked (much higher than typical dropout)
- `norm_pix_loss = True` — normalize each patch before computing MSE; prevents low-variance patches from dominating
- No labels used at any point — the supervision signal is pixel reconstruction

> **Why 75%?** Lower mask ratios make the task too easy — the model can infer masked patches from nearby context. 75% forces the model to understand global image structure, not just local texture.

In [ ]:
import os
os.makedirs('saved', exist_ok=True)

EPOCHS = 10
BATCH_SIZE = 128
LR = 1.5e-4

mean = [0.4914, 0.4822, 0.4465]
std  = [0.247,  0.243,  0.261]

train_tf = T.Compose([
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(mean, std),
])

train_ds = torchvision.datasets.CIFAR10('./data', train=True, transform=train_tf, download=True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05,
                               betas=(0.9, 0.95))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

model.train()
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for imgs, _ in train_loader:
        imgs = imgs.to(device)
        loss, _, _ = model(imgs)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()
    print(f'Epoch [{epoch+1}/{EPOCHS}]  Recon Loss: {epoch_loss/len(train_loader):.4f}')

torch.save(model.encoder.state_dict(), 'saved/mae_encoder.pt')
print('Saved encoder.')

## Linear Evaluation

Freeze the encoder and train a linear classifier on top. We pool the encoder outputs with a simple mean (global average pooling over patch tokens, excluding any cls token).

In [ ]:
# Freeze encoder entirely
model.encoder.eval()
for p in model.encoder.parameters():
    p.requires_grad = False

encoder_dim = 192
linear_clf = nn.Linear(encoder_dim, 10).to(device)
linear_opt = torch.optim.Adam(linear_clf.parameters(), lr=1e-3)

# Evaluation uses mask_ratio=0 trick: pass all patches through encoder
# Simpler: temporarily disable masking by setting mask_ratio=0
model.encoder.mask_ratio = 0.0

test_tf  = T.Compose([T.ToTensor(), T.Normalize(mean, std)])
train_tf2 = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(mean, std)])

clf_train = torchvision.datasets.CIFAR10('./data', train=True,  transform=train_tf2, download=False)
clf_test  = torchvision.datasets.CIFAR10('./data', train=False, transform=test_tf,   download=False)
clf_train_loader = DataLoader(clf_train, batch_size=256, shuffle=True,  num_workers=2)
clf_test_loader  = DataLoader(clf_test,  batch_size=256, shuffle=False, num_workers=2)

for ep in range(5):
    linear_clf.train()
    for imgs, labels in clf_train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        with torch.no_grad():
            # x_vis contains all patches when mask_ratio=0; mean pool
            x_vis, _, _ = model.encoder(imgs)
            feats = x_vis.mean(dim=1)  # global average pooling
        logits = linear_clf(feats)
        loss = F.cross_entropy(logits, labels)
        linear_opt.zero_grad()
        loss.backward()
        linear_opt.step()

    linear_clf.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in clf_test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            x_vis, _, _ = model.encoder(imgs)
            feats = x_vis.mean(dim=1)
            preds = linear_clf(feats).argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    print(f'Linear Eval Epoch {ep+1}/5 — Acc: {100*correct/total:.2f}%')

## Visualize Reconstructions

After training, we visualize three columns: **original → masked input → reconstruction**.

What to look for:
- The reconstruction should fill in masked patches with semantically plausible content
- It will not be pixel-perfect (not the goal) — the encoder learns *semantic* features, not texture memorization
- Blurry reconstructions are expected and fine — the useful representation lives in `h`, not the pixels

In [ ]:
import matplotlib.pyplot as plt

model.encoder.mask_ratio = 0.75  # restore masking for visualization
model.eval()

imgs_viz, _ = next(iter(DataLoader(
    torchvision.datasets.CIFAR10('./data', train=False, transform=test_tf),
    batch_size=8, shuffle=True
)))
imgs_viz = imgs_viz.to(device)

with torch.no_grad():
    loss, pred, mask = model(imgs_viz)

# Unpatchify prediction for display
p = model.patch_size
h = w = 32 // p

def unpatchify(patches, p, h, w, in_ch=3):
    # patches: (N, n_patches, p*p*C)
    N = patches.size(0)
    x = patches.reshape(N, h, w, p, p, in_ch)
    x = x.permute(0, 5, 1, 3, 2, 4)  # (N, C, h, p, w, p)
    x = x.reshape(N, in_ch, h*p, w*p)
    return x

pred_imgs = unpatchify(pred.cpu(), p, h, w)  # (N, 3, 32, 32)

# Denormalize for display
mean_t = torch.tensor(mean).view(3, 1, 1)
std_t  = torch.tensor(std).view(3, 1, 1)
orig_np  = (imgs_viz.cpu() * std_t + mean_t).clamp(0, 1).permute(0, 2, 3, 1).numpy()
pred_np  = (pred_imgs   * std_t + mean_t).clamp(0, 1).permute(0, 2, 3, 1).numpy()

# Build masked image: set masked patches to 0.5 (gray)
mask_expanded = mask.cpu().view(-1, h, w).unsqueeze(1)  # (N, 1, h, w)
mask_expanded = mask_expanded.repeat_interleave(p, dim=2).repeat_interleave(p, dim=3)  # (N, 1, 32, 32)
mask_np = mask_expanded.expand(-1, 3, -1, -1).permute(0, 2, 3, 1).numpy()
masked_np = orig_np.copy()
masked_np[mask_np.astype(bool)] = 0.5

N = 4
fig, axes = plt.subplots(3, N, figsize=(2*N, 6))
titles = ['Original', 'Masked (75%)', 'Reconstructed']
for row, (imgs_row, title) in enumerate(zip([orig_np, masked_np, pred_np], titles)):
    axes[row, 0].set_ylabel(title, fontsize=10)
    for col in range(N):
        axes[row, col].imshow(imgs_row[col])
        axes[row, col].axis('off')
plt.suptitle('MAE Reconstruction (CIFAR-10)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('saved/mae_reconstruction.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Reconstruction loss: {loss.item():.4f}')

## Summary

| Component | Detail |
|-----------|--------|
| **Backbone** | ViT-Small/6 (6 layers, embed_dim=192, 3 heads) adapted for CIFAR |
| **Patch size** | 4×4 → 64 patches per 32×32 image |
| **Mask ratio** | 75% of patches masked during pre-training |
| **Encoder input** | Visible patches only (25% = 16 patches) |
| **Decoder** | 4-layer Transformer, 128-dim, sees all 64 positions |
| **Loss** | MSE on masked patches (pixel space, per-patch normalized) |
| **Positional encoding** | Fixed 2D sinusoidal (no learned pos embed) |
| **Optimizer** | AdamW, lr=1.5e-4, weight_decay=0.05, β=(0.9, 0.95) |
| **No negatives** | Purely generative — no contrastive pairs needed |
| **Key insight** | Masking 75% forces non-trivial reconstruction; sparse encoder makes training fast |